# BTS Marketing Carrier On-Time Performance | Jan 2018 to Jan 2025

### Require libraries
The section below imports the required modules for the environment. Use `pip install -r requirements.txt` to install the required packages.

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.express as px
import pandas as pd
import warnings
from plotly.subplots import make_subplots
from ipyleaflet import Map, Polyline, CircleMarker, LayerGroup, basemaps, Marker
import plotly.graph_objects as go
warnings.filterwarnings("ignore")


### Utilities
The section below contains reusable utility functions used throughout this notebook.

In [2]:
def create_flow_plot(df, level1, level2, value):
    # build node list
    labels = pd.Index(pd.concat([df[level1], df[level2]]).unique())
    label_to_id = {label: i for i, label in enumerate(labels)}
    # map source/target ids
    df["source"] = df[level1].map(label_to_id)
    df["target"] =df[level2].map(label_to_id)
    #costume colors
    reason_colors = {
        "Carrier": "rgba(31,119,180,0.45)",
        "Weather": "rgba(255,127,14,0.45)",
        "NAS": "rgba(44,160,44,0.45)",
        "Security": "rgba(214,39,40,0.45)",
        "LateAircraft": "rgba(148,103,189,0.45)"}
    df["link_color"] = df[level2].map(reason_colors)
    fig = go.Figure(data=[
        go.Sankey(node=dict(pad=15,thickness=35,label=labels.tolist()),
                link=dict(source=df["source"],target=df["target"],
                 value=df[value],color=df["link_color"]))])
    fig.update_layout(font_size=12)
    return fig
def create_pie_plot(df, name_field, value_field, title=""):
    fig=px.pie(df, names=name_field, values=value_field, title=title)
    fig.update_traces(textinfo="percent+label", textposition="inside")
    return fig
def create_box_plot(df, x_field, y_field, color_field):
    fig= px.box(df,  x=x_field,  y=y_field, color=color_field,
                 points=False,log_y=True)
    fig.update_traces(line_width=0)
    return fig

def make_ipyleaflet_flight_map(plot_df ,min_count=0):
    plot_df = plot_df.dropna(subset=[
        "org_lat", "org_long", "dest_lat", "dest_long", "flight_count" ])
    plot_df = plot_df.groupby(["OriginStateName",  'DestStateName'],as_index=False
        ).agg(org_lat=("org_lat", "first"),org_long=("org_long", "first"),
        dest_lat=("dest_lat", "first"),dest_long=("dest_long", "first"),
        flight_count=("flight_count", "sum"))
    plot_df = plot_df[plot_df["flight_count"] >= min_count]
    center_lat = pd.concat([plot_df["org_lat"], plot_df["dest_lat"]]).mean()
    center_lon = pd.concat([plot_df["org_long"], plot_df["dest_long"]]).mean()
    m = Map(center=(center_lat, center_lon), zoom=5,basemap=basemaps.Esri.WorldStreetMap)
    max_count = plot_df["flight_count"].max()
    layers = []
    for _, row in plot_df.iterrows():
        weight =int(1 +15* (row["flight_count"] / max_count))
        layers.append(Polyline(locations=[
                    (row["org_lat"], row["org_long"]),
                    (row["dest_lat"], row["dest_long"])],
                weight=weight, color="navy"))
        layers.append(CircleMarker(location=(row["org_lat"], row["org_long"]), radius=2, color="green"))
        layers.append(CircleMarker(location=(row["dest_lat"], row["dest_long"]),radius=2, color="red"))
    m.add(LayerGroup(layers=layers))
    return m

months= ["January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"]

## Reading Input Dataset
The dataset used in this notebook was downloaded from the U.S. Department of Transportation website. You can access the dataset using the link below:
[Download link](https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=)
The dataset spans the years 2018 to 2025 and contains more than 56 million records in total. For the purposes of this report, 5 million records were selected randomly using the pandas sample() function.

In [3]:


data_path = r".\On_Time_Marketing_Carrier_On_Time_Performance_update_sample_v2.parquet"
main_df = pd.read_parquet(data_path, engine="fastparquet")

## get extract year and month names attribute from data field
main_df["year"] =main_df['FlightDate'].dt.year
main_df["month"] =main_df['FlightDate'].dt.month_name()
main_df["flight_count"]=1
main_df["airport_name"]=main_df["Origin"].astype(str)+"_"+main_df['OriginCityName'].astype(str)

main_df["month"] = pd.Categorical(main_df["month"],categories=months,ordered=True)
main_df =main_df.sort_values(by=['FlightDate','OriginStateName',"airport_name"], ascending=[False, True, True])
main_df['CancellationCode']=main_df['CancellationCode'].str.replace("D","Security")
main_df=main_df.sample(500000)
# summary_by_year=main_df["year"].value_counts().sort_index()
# from itables import show

# show(summary_by_year)


## Widgets setup
The section below includes ipywidgets components used to create interactive dashboards in followind sections.

In [4]:


##=== create dropdown options ====#
year_options = list(main_df['year'].unique())+["All"]
state_options = list(main_df['OriginStateName'].unique())+["All"]
airport_options = list(main_df["airport_name"].unique())+["All"]
month_options = months+["All"]
variable_options = list(main_df.select_dtypes(include="number").columns)

##=== Widgets ====#
year_dropdown = widgets.Dropdown(
    options=year_options,
    value=2025,
    description="Year"
)
state_dropdown = widgets.Dropdown(
    options=state_options,
    value="New York",
    description="State"
)
airport_dropdown = widgets.Dropdown(
    options=airport_options,
    value="JFK_New York, NY",
    description="Airport Name"
)

month_dropdown = widgets.Dropdown(
    options=month_options,
    value="January",
    description="Month"
)

variables_dropdown = widgets.Dropdown(
        options=variable_options,
        value="flight_count",
        description="Variable"
    )
    

## Explore the Dataset
The section below allows users to explore and compare a set of numerical attributes by filtering the data by year, state, and airport. You can use the left panel to explore the dataset and the panel on the right to compare attributes by year, state, and airport.

In [5]:
def create_panel():
    fig_widget = go.FigureWidget(layout=dict(
        template="plotly_dark",height=420,width=500))
    
    def update_dashboard(change=None):
        filtered_df = main_df.copy()
        # filter data based on options
        if year_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["year"] == year_dropdown.value]

        if state_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["OriginStateName"] == state_dropdown.value]
         
        if filtered_df.empty:
            print("No data found for selected filters")
            return
   
        filtered_df= filtered_df.groupby("month", as_index=False)[variables_dropdown.value]\
            .sum().sort_values("month")
        fig=create_pie_plot(filtered_df, "month",variables_dropdown.value,
                f'Summary of "{variables_dropdown.value}" by {state_dropdown.value} state and {year_dropdown.value} year')
   
        fig.update_layout(template="plotly_dark",height=400,
            width=500,margin=dict(l=20, r=20, t=60, b=20),showlegend=False )

        fig_widget.data = []
        for trace in fig.data:
            fig_widget.add_trace(trace)
        fig_widget.layout = fig.layout

    for w in [year_dropdown, state_dropdown, variables_dropdown]:
        w.observe(update_dashboard, names="value")

    controls = widgets.HBox(
        [year_dropdown, state_dropdown, variables_dropdown],
        layout=widgets.Layout(
            gap="20px",align_items="center", flex_flow="row wrap")
    )

    panel = widgets.VBox(
        [controls, fig_widget],
        layout=widgets.Layout( width="50%",gap="15px")
    )

    update_dashboard()
    return panel

left_panel = create_panel()
right_panel = create_panel()

dashboard = widgets.HBox(
    [left_panel, right_panel],
    layout=widgets.Layout(width="100%",gap="20px",align_items="flex-start")
)

display(dashboard)

## What Causes the Flight Delays 
The section below allows users to explore and understand flight delays and their causes by filtering the data across different time periods and locations.

In [6]:
def update_airport_options2(change=None):
    filtered_airports =main_df.loc[main_df["OriginStateName"] == state_dropdown.value, "airport_name"].unique().tolist()
    airport_dropdown.options = ["All"] + sorted(filtered_airports)
    airport_dropdown.value = "All"


output2 = widgets.Output()
    
def update_dashboard2(change=None):
    with output2:
        clear_output(wait=True)
        filtered_df = main_df.copy()

        # Year filter
        if year_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["year"] == year_dropdown.value]

        # state filter
        if state_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df['OriginStateName'] == state_dropdown.value]

        # airport filter
        if airport_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["airport_name"] == airport_dropdown.value]

         # month filter
        if month_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["month"] == month_dropdown.value]

        if filtered_df.empty:
            print("No data found for selected filters.")
            return
        
        ## delay feilds
        delay_fields=[ 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay','LateAircraftDelay']
        ## overall on time versus delay flights
        filtered_df['DepDel15']=filtered_df['DepDel15'].fillna(-1)
        delay_df_count=filtered_df['DepDel15'].value_counts().reset_index()
        delay_df_count['DepDel15_cat']=delay_df_count['DepDel15'].map({0:"On time", 1:"Delay",-1:"Canceled"})
        fig1=create_pie_plot(delay_df_count, 'DepDel15_cat',"count")
    
        # Delay reasons
        delay_df=filtered_df[filtered_df["DepDel15"]==1][delay_fields]
        delay_df=delay_df.melt(var_name="delay_reason", value_name="delay_time")
        delay_df=delay_df[(delay_df["delay_time"].notnull())&(delay_df["delay_time"]>0)]
        delay_df["delay_reason"]=delay_df["delay_reason"].str.replace("Delay","")
        
        delay_df_count=delay_df["delay_reason"].value_counts().reset_index()
        fig2=create_pie_plot(delay_df_count, "delay_reason","count")
    

        delay_df_count=filtered_df['CancellationCode'].value_counts().reset_index()
        fig3=create_pie_plot(delay_df_count, 'CancellationCode',"count")
  
        # delay time distribution
        fig4 =create_box_plot(delay_df, "delay_reason", "delay_time", "delay_reason")

        ## delay by months
        delay_df=filtered_df[['DepDel15',"month"]+delay_fields]
        delay_df.loc[:,delay_fields] = delay_df.loc[:, delay_fields].mask(delay_df.loc[:,delay_fields] < 15)
        delay_df=delay_df[delay_df['DepDel15']==1]
        delay_df= pd.melt(delay_df, id_vars=['DepDel15',"month"], var_name='delay_reason', value_name='delay_time')
        delay_df=delay_df[delay_df["delay_time"].notnull()]
        delay_df=delay_df[["month",'delay_reason']].value_counts().reset_index(name="delay_count")
        delay_df['delay_reason']=delay_df['delay_reason'].str.replace("Delay","")
        fig5=create_flow_plot(delay_df, "month",'delay_reason' ,"delay_count")

        ## combine plots
        combined_fig = make_subplots(rows=3,cols=2,
            specs=[[{"type": "domain"}, {"type": "domain"}],
                [{"type": "domain"},{"type": "xy"}],
                 [{"type": "domain", "colspan": 2}, None]],
      
            subplot_titles=("Flight Status","Cancellation Reasons", "Delay Reasons",
                            "Delay time distribution","Delay Reasons by Months" ),     
        )

        for trace in fig1.data:
            combined_fig.add_trace(trace, row=1, col=1)
        for trace in fig2.data:
            combined_fig.add_trace(trace, row=2, col=1)
        for trace in fig3.data:
            combined_fig.add_trace(trace, row=1, col=2)
        for trace in fig4.data:
            combined_fig.add_trace(trace, row=2, col=2)
        for trace in fig5.data:
            combined_fig.add_trace(trace, row=3, col=1)

        combined_fig.update_xaxes(title_text="Delay Reasons", row=2, col=1)
        combined_fig.update_yaxes(title_text="Delay in minutes ( log scale)",
                                    type="log", row=2, col=2,  title_font=dict(size=10))

        combined_fig.update_layout(height=900,width=900,
            template="plotly_dark",showlegend=False
        )

        display(combined_fig)

# Widget events
year_dropdown.observe(update_dashboard2, names="value")
state_dropdown.observe(update_airport_options2, names="value")
state_dropdown.observe(update_dashboard2, names="value")
airport_dropdown.observe(update_dashboard2, names="value")
month_dropdown.observe(update_dashboard2, names="value")
# Layout
filters2 = widgets.HBox([year_dropdown, state_dropdown,airport_dropdown,month_dropdown])
update_airport_options2()
display(filters2)
display(output2)
update_dashboard2()

Output()

## Explore Flight patters
The section below allows users to explore flight patterns  across different time periods and locations.

In [7]:
def update_airport_options3(change=None):
    filtered_airports =main_df.loc[main_df["OriginStateName"] == state_dropdown.value, "airport_name"].unique().tolist()
    airport_dropdown.options = ["All"] + sorted(filtered_airports)
    airport_dropdown.value = "All"

output3 = widgets.Output()

def update_dashboard3(change=None):
    with output3:
        clear_output(wait=True)
        filtered_df = main_df.copy()
        # Year filter
        filtered_df = filtered_df[filtered_df["year"] == year_dropdown.value]
        # state filter
        filtered_df = filtered_df[filtered_df['OriginStateName'] == state_dropdown.value]
        # airport filter
        if airport_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["airport_name"] == airport_dropdown.value]
         # month filter
        filtered_df = filtered_df[filtered_df["month"] == month_dropdown.value]

        if filtered_df.empty:
            print("No data found for selected filters.")
            return   
        m = make_ipyleaflet_flight_map(filtered_df , min_count=10)
        display(m)

# Widget events
year_dropdown.observe(update_dashboard3, names="value")
state_dropdown.observe(update_airport_options3, names="value")
state_dropdown.observe(update_dashboard3, names="value")
airport_dropdown.observe(update_dashboard3, names="value")
month_dropdown.observe(update_dashboard3, names="value")
# Layout
filters3 = widgets.HBox([year_dropdown, state_dropdown,month_dropdown, airport_dropdown])
update_airport_options3()
display(filters3)
display(output3)
update_dashboard3()

Output()